# Making Plots of the data

Imports:

In [1]:
import sys
from pathlib import Path

root_dir = Path().resolve().parent
sys.path.append(str(root_dir))

from lib.types.report import StudyReport, BrainReport, StudyReportCollection
from lib.types.config import ParamsConfig

report_dir = root_dir / "reports"

In [3]:
d = {}
for i in range(7, 11):
    file = report_dir / f"v{i}_pet_llm_smollm2-1.7b-q8_0.json"
    report = StudyReport.model_validate_json(file.read_text())
    report = StudyReportCollection.average_study(report)
    
    oob = report.simulation_config.brain.thoughts.out_of_bounds_message

    thought_loop_ratio = float("inf")
    best = None
    for trial in report.trials:
        config, result = trial.params, trial.report
        if result.iterations < 10:
            continue

        ratio_temp = result.thought_loops / result.iterations
        if ratio_temp < thought_loop_ratio:
            thought_loop_ratio = ratio_temp
            best = trial

            d[oob] = ratio_temp

    assert best is not None, "couldn't get trial"

    avg_list: list[BrainReport] = []
    for trial in report.trials:
        trial.params.seed = None
        if trial.params == best.params:
            avg_list.append(trial.report)

    
    print(f"best for {oob}: {sum( [( x.iterations) for x in avg_list] )}")



best for You can't leave the tank! Try a coordinate inside ({}, {}).: 24
best for You can't leave the tank! Ensure x coordinate is between 0 and {}, and y coordinate is between 0 and {}.: 28
best for You can't leave the tank!: 38
best for : 43
